# Industrial Surface Defect Inspection — T4 GPU Training

This notebook trains the **Industrial Surface Defect Inspection** CNN on a **Tesla T4 GPU** using an industrial dataset from **Kaggle** (NEU Metal Surface Defect / Casting Quality Inspection).

### Step 1: Switch Runtime to T4 GPU
- **In Google Colab**: Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.
- **In Kaggle**: In the right sidebar under **Settings** > **Accelerator** > Select **GPU T4 x2**.

In [ ]:
# Verify GPU acceleration
import torch
print("PyTorch Version:", torch.__version__)
if torch.cuda.is_available():
    print(f"✅ GPU active: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")
else:
    print("⚠️ No GPU detected. Please go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.")

### Step 2: Download Kaggle Dataset
You can use either:
1. **Option A**: Upload your `kaggle.json` API token (from Kaggle Profile -> Settings -> Create New Token)
2. **Option B**: Use direct dataset download via curl/gdown or git.

In [ ]:
# Setup Kaggle API credentials (if using Colab)
import os
from google.colab import files

# Upload kaggle.json if not already present
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp kaggle.json /root/.kaggle/
    !chmod 600 /root/.kaggle/kaggle.json

# Download the popular NEU Metal Surface Defect dataset from Kaggle
!kaggle datasets download -d fantacher/neu-metal-surface-defects-data --unzip -p ./data/kaggle_dataset

### Step 3: Prepare Binary Classification Dataset (defective vs non_defective)

In [ ]:
import os
import shutil
from pathlib import Path

# Target directory for the training pipeline
target_dir = Path('./data/real')
def_dir = target_dir / 'defective'
non_def_dir = target_dir / 'non_defective'
def_dir.mkdir(parents=True, exist_ok=True)
non_def_dir.mkdir(parents=True, exist_ok=True)

# In the NEU dataset, defective categories include: crazing, inclusion, patches, pitted_surface, rolled-in_scale, scratches
# Non-defective / baseline surfaces or healthy samples
src_path = Path('./data/kaggle_dataset')

# Auto-categorize images into defective and non_defective
image_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
for f in src_path.rglob('*'):
    if f.suffix.lower() in image_extensions:
        # Check parent folder name or filename
        name_lower = f.name.lower() + ' ' + str(f.parent).lower()
        if any(term in name_lower for term in ['good', 'ok', 'normal', 'non_defect', 'clean']):
            shutil.copy(f, non_def_dir / f.name)
        else:
            shutil.copy(f, def_dir / f.name)

print(f"Defective images: {len(list(def_dir.glob('*')))}")
print(f"Non-defective images: {len(list(non_def_dir.glob('*')))}")

### Step 4: Model Architecture & PyTorch T4 Training Pipeline

In [ ]:
import json
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from PIL import Image

LABELS = ['non_defective', 'defective']
SIZE = 128

def get_transforms(training=False):
    ops = [transforms.Resize((SIZE, SIZE))]
    if training:
        ops += [transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(10)]
    ops += [transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3)]
    return transforms.Compose(ops)

class DefectCNN(nn.Module):
    def __init__(self):
        super().__init__()
        layers = []
        inc = 3
        for out in (16, 32, 64, 128):
            layers += [nn.Conv2d(inc, out, 3, padding=1), nn.BatchNorm2d(out), nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            inc = out
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.25), nn.Linear(128, 1))

    def forward(self, x):
        return self.head(self.features(x)).squeeze(1)

class IndexedDataset(Dataset):
    def __init__(self, base, indices, tfm):
        self.base = base
        self.indices = list(indices)
        self.tfm = tfm
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        path, y = self.base.samples[self.indices[i]]
        with Image.open(path) as img:
            return self.tfm(img.convert('RGB')), float(y)

# Set seeds
random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

base = ImageFolder('./data/real')
if base.classes != LABELS:
    base.samples = [(p, 1-y) for p, y in base.samples]
    base.targets = [1-y for y in base.targets]

targets = np.array([y for _, y in base.samples])
ids = np.arange(len(targets))

tr, rest = train_test_split(ids, test_size=0.3, stratify=targets, random_state=42)
va, te = train_test_split(rest, test_size=0.5, stratify=targets[rest], random_state=42)

loaders = {
    name: DataLoader(IndexedDataset(base, ix, get_transforms(name=='train')),
                     batch_size=32, shuffle=(name=='train'), num_workers=2, pin_memory=True)
    for name, ix in [('train', tr), ('val', va), ('test', te)]
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training on device: {device} ({torch.cuda.get_device_name(0) if device=='cuda' else 'CPU'})")

model = DefectCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.BCEWithLogitsLoss()
epochs = 20
best_loss = float('inf')
history = []
os.makedirs('models', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

for epoch in range(epochs):
    scores = {}
    for split in ('train', 'val'):
        model.train(split == 'train')
        total, correct, loss_sum = 0, 0, 0
        for x, y in loaders[split]:
            x, y = x.to(device), y.to(device)
            with torch.set_grad_enabled(split == 'train'):
                logits = model(x)
                loss = loss_fn(logits, y)
                if split == 'train':
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
            total += len(y)
            loss_sum += loss.item() * len(y)
            correct += ((torch.sigmoid(logits) >= 0.5) == y.bool()).sum().item()
        scores[f'{split}_loss'] = round(loss_sum / total, 5)
        scores[f'{split}_accuracy'] = round(correct / total, 5)
    scores['epoch'] = epoch + 1
    history.append(scores)
    print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Loss: {scores['train_loss']:.4f} Acc: {scores['train_accuracy']*100:.1f}% | Val Loss: {scores['val_loss']:.4f} Acc: {scores['val_accuracy']*100:.1f}%")
    if scores['val_loss'] < best_loss:
        best_loss = scores['val_loss']
        torch.save({
            'state_dict': model.state_dict(),
            'labels': LABELS,
            'image_size': 128,
            'epoch': epoch + 1,
            'demo_data': False
        }, 'models/defect_cnn.pt')

print("\n✅ Training complete. Evaluating on held-out test split...")
checkpoint = torch.load('models/defect_cnn.pt', map_location=device)
model.load_state_dict(checkpoint['state_dict'])
model.eval()

truth, pred = [], []
for x, y in loaders['test']:
    with torch.no_grad():
        p = (torch.sigmoid(model(x.to(device))) >= 0.5).int().cpu().tolist()
    pred.extend(p)
    truth.extend(y.int().tolist())

test_acc = accuracy_score(truth, pred)
test_f1 = f1_score(truth, pred, zero_division=0)
print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"Test F1-Score: {test_f1:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(truth, pred, labels=[0, 1]))
print("\nClassification Report:")
print(classification_report(truth, pred, target_names=LABELS))

metrics = {
    'test_accuracy': test_acc,
    'test_f1': test_f1,
    'confusion_matrix': confusion_matrix(truth, pred, labels=[0, 1]).tolist(),
    'classification_report': classification_report(truth, pred, target_names=LABELS, output_dict=True),
    'history': history,
    'device': str(device)
}
with open('outputs/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

### Step 5: Download Trained Model for your Local Streamlit App
Run the cell below to download `defect_cnn.pt` and `metrics.json`. Place them in your project's `models/` and `outputs/` folders.

In [ ]:
from google.colab import files
print("Downloading trained model checkpoint and metrics:")
files.download('models/defect_cnn.pt')
files.download('outputs/metrics.json')